In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
pip install sentence-transformers

<hr>

# Projektbeitrag (Beispiel)

#### Die Lernenden erstellen aus dem Zusammenwirken verschiedener Sprachmodelle einen möglichen Workflow. Dies kann grundsätzlich auch mit einem Sprachmodell geschehen, wenn im Vorfeld entsprechende Anfragen generiert wurden und das Sprachmodell hierfür geeignet ist.

## Ziel: Niederschwellige Darstellung von Retrieval Augmented Generation (RAG)

> Unter Retrieval-Augmented Generation (RAG) versteht man ein Softwaresystem, welches Information Retrieval mit einem Large Language Model kombiniert. Eine Abfrage, welche an das System gestellt wird, kann hierbei auf Informationen aus (externen) Informationsquellen, Datenbanken oder dem World Wide Web zugreifen statt nur auf die Trainingsdaten des Modells. Dies erhöht die Genauigkeit und Robustheit der generierten Inhalte, indem es die Modelle mit aktuellen und spezifischen Informationen versorgt. Typische Anwendungsfälle sind der Zugriff von Chatbots auf interne (Unternehmens-)Daten oder die Bereitstellung von Sachinformationen, die ausschließlich aus verlässlichen Quellen stammen sollen.
> Quelle: https://de.wikipedia.org/wiki/Retrieval-Augmented_Generation

### Voraussetzungen:

#### Die Lernenden müssen die Materialien zu Embeddings und dem Chatbot kennengelernt haben.

<br>

<hr>

In [ ]:
from sentence_transformers import SentenceTransformer # Import der Funktion pipeline des Moduls transformers
from transformers import pipeline # Import der Funktion pipeline des Moduls transformers

### Festlegung der Datenquellen deren Inhalt nicht in einem Sprachmodell enthalten sind. 
### In der Praxis stammen diese aus externen Quellen, wie z.B. PDF-Dokumenten oder Webseiten

In [ ]:

datenquelle= ["Die KI-Schule hat werktags von 8-15 Uhr geöffnet.", "Das Kollegium besteht aus 80 Lehrkräften.","Sie erreichen uns unter folgender Nummer: 0911-123456.","Die Teilnahme an den Kursen ist kostenlos."]

### Das Modell errechnet jetzt die Vektordarstellung (Embedding) einer jeden Datenquelle

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
quellenEmbeddings = model.encode(datenquelle)
quellenEmbeddings

In [ ]:
userprompt="Bitte teilen Sie mir mit, wie ich Sie kontaktieren kann."

In [ ]:
userpromptEmbedding=model.encode(userprompt) # Die nummerische Darstellung (Embedding) des Userprompts wird erstellt.
# userpromptEmbedding # Kommentar löschen, falls das Embedding angezeigt werden soll.

### Berechnung der Ähnlichkeiten zwischen Userprompt und Inhalten der Datenquellen

In [ ]:
# Das Dictionary wird dynamisch erstellt
ergebnis_dict = {
    datenquelle[i]: model.similarity(userpromptEmbedding, quellenEmbeddings[i]).item()
    for i in range(len(quellenEmbeddings))
}

In [ ]:
for key, value in ergebnis_dict.items():
    print(f"{key} : Ähnlichkeit: {value}.")

### Erstellung der Antwort aufgrund des Userprompts und der wahrscheinlichsten Datenquelle

In [ ]:
max_key = max(ergebnis_dict, key=ergebnis_dict.get)
print("Wahrscheinlichste Datenquelle: " + max_key)
antwort = max_key

pipe = pipeline("text-generation", model="LiquidAI/LFM2-350M", device=0, max_new_tokens=500, temperature=0.01)
messages = [
    {"role": "system", "content": "Du bist ein nützlicher Chatbot und bekommst folgende Anfrage des Users: #"+userprompt+"#. Die Antwort auf die Frage lautet:#"+antwort+"#. Gib die Antwort auf die Frage in eigenen Worten passend zur Anfrage wieder. Fasse Dich kurz!"}
]
antwortprompt = pipe(messages)
antwortprompt[0]['generated_text'][-1]['content']